# CSV Comparison Tool
Place this notebook in the parent folder containing all your CSV directories. Run Cell 1 then Cell 2.

In [8]:
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

files = []

def scan():
    global files
    files = sorted(Path(".").rglob("*.csv"))
    for file in files:
        if "ipnyb" in str(file):
            files.pop(index(file))
    opts=[str(f) for f in files]
    dd1.options=opts
    dd2.options=opts
    if opts:
        dd1.value=opts[0]
        dd2.value=opts[min(1,len(opts)-1)]

def compare(_=None):
    out.clear_output()
    if not dd1.value or not dd2.value: return
    a=pd.read_csv(dd1.value)
    b=pd.read_csv(dd2.value)
    with out:
        print("A:",dd1.value)
        print("B:",dd2.value)
        print()
        print(f"A shape: {a.shape}    B shape: {b.shape}")
        onlyA=sorted(set(a.columns)-set(b.columns))
        onlyB=sorted(set(b.columns)-set(a.columns))
        shared=sorted(set(a.columns)&set(b.columns))
        print("Only in A:",onlyA)
        print("Only in B:",onlyB)
        print("Shared columns:",len(shared))
        display(widgets.HBox([
            widgets.Output(layout={'border':'1px solid gray','width':'50%'}),
            widgets.Output(layout={'border':'1px solid gray','width':'50%'})
        ]))


In [9]:
# Run this cell.
from IPython.display import display
left=widgets.Dropdown(description="File A",layout=widgets.Layout(width='600px'))
right=widgets.Dropdown(description="File B",layout=widgets.Layout(width='600px'))
dd1,dd2=left,right
btn_refresh=widgets.Button(description="Refresh")
btn_compare=widgets.Button(description="Compare")
out=widgets.Output()

btn_refresh.on_click(scan)
btn_refresh.on_click(compare)
btn_compare.on_click(compare)

display(widgets.HBox([left,right]))
display(widgets.HBox([btn_refresh,btn_compare]))
display(out)

scan()

def compare(_=None):
    out.clear_output()
    a=pd.read_csv(dd1.value)
    b=pd.read_csv(dd2.value)
    with out:
        print("=== SUMMARY ===")
        print(dd1.value)
        print(dd2.value)
        print(f"A: {a.shape}   B: {b.shape}\n")
        onlyA=sorted(set(a.columns)-set(b.columns))
        onlyB=sorted(set(b.columns)-set(a.columns))
        print("Only in A:",onlyA)
        print("Only in B:",onlyB)
        shared=[c for c in a.columns if c in b.columns]
        print("\nPreview:")
        display(widgets.HBox([
            widgets.Output(layout=widgets.Layout(width='50%')),
            widgets.Output(layout=widgets.Layout(width='50%'))
        ]))
        # Simpler display:
        display(pd.concat([a.head(20)],axis=1))
        display(pd.concat([b.head(20)],axis=1))
        if list(a.columns)==list(b.columns):
            if a.shape==b.shape:
                diff=a.compare(b, keep_shape=False, keep_equal=False)
                print("\nDifferences:")
                if diff.empty:
                    print("Files are identical.")
                else:
                    display(diff.head(100))
            else:
                print("\nColumns match but row counts differ.")
compare()


Output()